In [1]:
import torch
import os
import pickle
import torch.nn.functional as F
import numpy as np
try:
    from kmeans_pytorch import kmeans
except Exception:
    kmeans = None

In [2]:
dataset = "LastFM"
user_emb = pickle.load(open(os.path.join(dataset+"/handled/", "usr_emb_np.pkl"), "rb"))

In [3]:
# 将 numpy 数组高效转换为张量并选择设备
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
try:
    user_emb = torch.from_numpy(user_emb).float().to(device)
except torch.cuda.OutOfMemoryError:
    device = torch.device("cpu")
    user_emb = torch.from_numpy(user_emb).float()
user_emb = F.normalize(user_emb, dim=1)

# user_emb=torch.tensor(user_emb, dtype=torch.float32)

In [4]:
def _euclidean_distances(x, centers):
    x_norm = (x**2).sum(dim=1, keepdim=True)
    c_norm = (centers**2).sum(dim=1).unsqueeze(0)
    return x_norm + c_norm - 2.0 * (x @ centers.T)

def kmeans_plus_plus_init(x, k, seed=42):
    g = torch.Generator(device=device)
    g.manual_seed(seed)
    n = x.shape[0]
    idx0 = torch.randint(0, n, (1,), generator=g, device=device).item()
    centers = x[idx0:idx0+1].clone()
    for _ in range(1, k):
        dists = _euclidean_distances(x, centers)
        min_dist, _ = dists.min(dim=1)
        probs = min_dist.clamp_min(1e-12)
        probs = probs / probs.sum()
        next_idx = torch.multinomial(probs, 1, generator=g).item()
        centers = torch.cat([centers, x[next_idx:next_idx+1]], dim=0)
    return centers

def run_kmeans(x, num_clusters, max_iter=50, tol=1e-4):
    centers = kmeans_plus_plus_init(x, num_clusters)
    for _ in range(max_iter):
        dists = _euclidean_distances(x, centers)
        cluster_ids = dists.argmin(dim=1)
        new_centers = torch.zeros_like(centers)
        for i in range(num_clusters):
            mask = (cluster_ids == i)
            if mask.any():
                new_centers[i] = x[mask].mean(dim=0)
            else:
                # 罕见：簇为空，按 k-means++ 以距离平方作为概率重新选择一个中心
                min_dist, _ = dists.min(dim=1)
                probs = min_dist.clamp_min(1e-12)
                probs = probs / probs.sum()
                far_idx = torch.multinomial(probs, 1).item()
                new_centers[i] = x[far_idx]
        shift = (new_centers - centers).pow(2).sum().sqrt()
        centers = new_centers
        if shift.item() <= tol:
            break
    cluster_centers = F.normalize(centers, dim=1).to(device)
    cluster_ids_x = cluster_ids.to(device)
    return cluster_ids_x, cluster_centers

In [5]:
cluster_ids_x, cluster_centers = run_kmeans(user_emb, num_clusters=50)
labels = cluster_ids_x.detach().cpu().numpy()
centers_np = cluster_centers.detach().cpu().numpy()

# 使用 FCM 计算软 membership（scores），作为静态门控的早期资料
def fcm_membership(X, centers, m=2.0, max_iter=30, eps=1e-5):
    N, D = X.shape
    K = centers.shape[0]
    device_local = X.device
    batch_size = min(1024, N)
    U = X.new_zeros((N, K))
    def fill_U(centers_local):
        i = 0
        b = batch_size
        while i < N:
            j = min(i + b, N)
            xb = X[i:j]
            try:
                d2b = _euclidean_distances(xb, centers_local).clamp_min(1e-12)
                inv = d2b.reciprocal()
                ub = inv / inv.sum(dim=1, keepdim=True)
                U[i:j] = ub
                i = j
            except torch.cuda.OutOfMemoryError:
                if device_local.type == 'cuda' and b > 128:
                    b = max(b // 2, 128)
                    torch.cuda.empty_cache()
                else:
                    xb_cpu = xb.cpu()
                    centers_cpu = centers_local.cpu()
                    d2b = _euclidean_distances(xb_cpu, centers_cpu).clamp_min(1e-12)
                    inv = d2b.reciprocal()
                    ub = inv / inv.sum(dim=1, keepdim=True)
                    U[i:j] = ub.to(device_local)
                    i = j
    fill_U(centers)
    for _ in range(max_iter):
        U_m = U.pow(m)
        numer = U_m.transpose(0,1) @ X
        denom = U_m.sum(dim=0).unsqueeze(1) + 1e-12
        new_centers = numer / denom
        fill_U(new_centers)
        if (new_centers - centers).pow(2).sum().sqrt().item() < eps:
            centers = new_centers
            break
        centers = new_centers
    return centers, U

centers_fcm, U = fcm_membership(user_emb, cluster_centers, m=2.0, max_iter=30)
# 替换为 FCM 更新后的中心（保持维度与数量不变）
cluster_centers = F.normalize(centers_fcm, dim=1)
centers_np = cluster_centers.detach().cpu().numpy()
def sinkhorn_balance(U, iters=5, eps=1e-6):
    N, K = U.shape
    w = U + eps
    target = (N / max(K, 1))
    for _ in range(max(iters, 1)):
        w = w / (w.sum(dim=1, keepdim=True) + 1e-8)
        col = w.sum(dim=0, keepdim=True) + 1e-8
        w = w / col
        w = w * target
    w = w / (w.sum(dim=1, keepdim=True) + 1e-8)
    return w
U_bal = sinkhorn_balance(U)
scores = U_bal.detach().cpu().numpy()

import pickle, os
out = {"labels": labels, "centers": centers_np, "scores": scores}
pickle.dump(out, open(os.path.join(dataset+"/handled/", "user_label.pkl"), "wb"))


In [6]:
user_emb.dtype

torch.float32

In [7]:
# 计算相似度矩阵
# S = torch.matmul(user_emb.cuda(), cluster_centers.T)

# 计算相似度矩阵（用户与中心均单位化，近似余弦相似度）
user_norm = F.normalize(user_emb, dim=1)
S = user_norm @ cluster_centers.T

In [8]:
# 按行归一化相似度矩阵，加入数值稳定项
S_min, _ = torch.min(S, dim=1, keepdim=True)
S_max, _ = torch.max(S, dim=1, keepdim=True)
# S_normalized = (S - S_min) / (S_max - S_min)
# 确保归一化后的值在 (0, 1) 之间
# S_normalized = torch.clamp(S_normalized, 0, 1)

denom = (S_max - S_min).clamp_min(1e-8)
S_normalized = torch.clamp((S - S_min) / denom, 0, 1)

In [9]:
S_normalized.mean()

tensor(0., device='cuda:0')

In [10]:
## Save llm embedding based similar users
# pickle.dump(S_normalized, open(os.path.join(dataset+"/handled/", "user_label.pkl"), "wb"))

# 保存软标签矩阵与聚类结果（scores、cluster_ids、centers）
out = {
    'scores': S_normalized.detach().cpu(),
    'cluster_ids': cluster_ids_x.detach().cpu(),
    'centers': cluster_centers.detach().cpu(),
}
pickle.dump(out, open(os.path.join(dataset+"/handled/", "user_label.pkl"), "wb"))